In [ ]:
crypto-volatility/
│
├── data/
│   └── dataset.csv
│
├── notebooks/
│   └── eda.ipynb
│
├── src/
│   ├── preprocess.py
│   ├── feature_engineering.py
│   ├── train.py
│   ├── predict.py
│
├── models/
│   └── model.pkl
│
├── app.py   (Streamlit app)
├── requirements.txt
└── README.md

In [26]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def load_data(path):
    df = pd.read_csv(path)
    print("DataFrame columns:", df.columns.tolist()) # Added for debugging
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(by='date')
    return df

def clean_data(df):
    df = df.drop_duplicates()
    df = df.ffill() # Updated to use ffill() directly
    return df

def scale_features(df, feature_cols):
    scaler = MinMaxScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])
    return df, scaler


In [20]:
import pandas as pd
import numpy as np

def create_features(df):
    # Daily returns
    df['return'] = (df['close'] - df['open']) / df['open']

    # Rolling volatility
    df['volatility_7'] = df['return'].rolling(window=7).std()
    df['volatility_14'] = df['return'].rolling(window=14).std()

    # Moving averages
    df['ma_7'] = df['close'].rolling(window=7).mean()
    df['ma_14'] = df['close'].rolling(window=14).mean()

    # Liquidity
    df['volume_marketcap_ratio'] = df['volume'] / df['marketCap']

    df = df.dropna()
    return df

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import pickle
import numpy as np # Added for handling infinite values
import os # Added to handle directory creation

# The functions load_data, clean_data, and create_features are already defined
# in previous executed cells and are available in the global scope.
# Therefore, the import statements below are not needed.
# from preprocess import load_data, clean_data
# from feature_engineering import create_features

# Load and prepare data
df = load_data('/content/dataset.csv')
df = clean_data(df)
df = create_features(df)

# Handle infinite values introduced by feature engineering
# Replace inf and -inf with NaN, then drop rows with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Features & target
features = ['open', 'high', 'low', 'close', 'volume', 'marketCap',
            'ma_7', 'ma_14', 'volume_marketcap_ratio']

target = 'volatility_7'

X = df[features]
y = df[target]

# Time-based split
split = int(len(df) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Model
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

# Evaluation
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)

# Save model
model_dir = '/content/models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model, f)


DataFrame columns: ['Unnamed: 0', 'open', 'high', 'low', 'close', 'volume', 'marketCap', 'timestamp', 'crypto_name', 'date']


/tmp/ipykernel_5644/2912455392.py:13: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')


RMSE: 0.2987900474593049


In [27]:
import pickle
import pandas as pd

def load_model():
    with open('../models/model.pkl', 'rb') as f:
        return pickle.load(f)

def predict(input_df):
    model = load_model()
    return model.predict(input_df)

In [29]:
!pip install streamlit
import streamlit as st
import pandas as pd
import pickle

model = pickle.load(open('models/model.pkl', 'rb'))

st.title("Crypto Volatility Predictor")

open_price = st.number_input("Open")
high = st.number_input("High")
low = st.number_input("Low")
close = st.number_input("Close")
volume = st.number_input("Volume")
market_cap = st.number_input("Market Cap")

if st.button("Predict"):
    input_data = pd.DataFrame([[open_price, high, low, close, volume, market_cap, 0, 0, 0]],
                              columns=['Open','High','Low','Close','Volume','Market Cap',
                                       'ma_7','ma_14','volume_marketcap_ratio'])

    prediction = model.predict(input_data)
    st.write("Predicted Volatility:", prediction[0])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 57.4 MB/s eta 0:00:00


2026-03-30 08:31:33.769 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-30 08:31:33.872 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-30 08:31:33.873 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-30 08:31:33.874 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-30 08:31:33.876 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-30 08:31:33.878 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-30 08:31:33.880 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-30 08:31:33.881 Thread 'MainThread': mi